# Machine Doctor — Deep Learning Add-on
## Step 6: Train the Final Model & Export for Deployment

Step 5's leave-one-load-out cross-validation already proved this architecture generalizes across load conditions (0.999 pooled accuracy). That was a *validation* exercise -- each fold deliberately withheld data to test honestly. Now that we trust the approach, the model we actually ship should be trained on **all available real data**, holding nothing back.

This notebook trains that final model and exports everything Django needs: the trained weights, the class names (so predictions map back to real fault names), and the window size (so the Django side knows how to prepare its input).

In [ ]:
!pip install -q kagglehub
import kagglehub, os, glob, re, json
import numpy as np
import scipy.io
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_path = kagglehub.dataset_download("esraakhaled299/cwru-data")
print("Device:", device, "| Dataset at:", dataset_path)

In [ ]:
CLASS_NAMES = ["Normal", "Ball", "InnerRace", "OuterRace"]
WINDOW_SIZE = 1024
STRIDE = 512

all_files = glob.glob(os.path.join(dataset_path, "**", "*.mat"), recursive=True)
de_files = [f for f in all_files if "12k_DE" in f or f"{os.sep}Normal{os.sep}" in f]

labeled_files = []
for f in de_files:
    class_name = None
    for c in CLASS_NAMES:
        if c in f:
            class_name = c
            break
    if class_name is not None:
        labeled_files.append((f, class_name))

print(f"Using ALL {len(labeled_files)} files (no held-out split -- this is the deployment model)")
for c in CLASS_NAMES:
    print(f"  {c}: {sum(1 for _, cn in labeled_files if cn == c)} files")

In [ ]:
def load_de_signal(filepath):
    mat = scipy.io.loadmat(filepath)
    key = [k for k in mat.keys() if "DE_time" in k][0]
    return mat[key].flatten()

def make_windows(file_label_list):
    X, y = [], []
    for filepath, class_name in file_label_list:
        signal = load_de_signal(filepath)
        for start in range(0, len(signal) - WINDOW_SIZE, STRIDE):
            window = signal[start:start + WINDOW_SIZE]
            window = (window - window.mean()) / (window.std() + 1e-8)
            X.append(window)
            y.append(CLASS_NAMES.index(class_name))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

X_all, y_all = make_windows(labeled_files)
print(f"Total training windows: {X_all.shape}")

In [ ]:
class VibrationCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=64, stride=2, padding=32),
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=32, stride=2, padding=16),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=16, stride=2, padding=8),
            nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 32), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = VibrationCNN(num_classes=len(CLASS_NAMES)).to(device)

class_counts = np.array([np.sum(y_all == i) for i in range(len(CLASS_NAMES))])
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = (class_weights / class_weights.sum() * len(CLASS_NAMES)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

loader = DataLoader(
    TensorDataset(torch.tensor(X_all).unsqueeze(1), torch.tensor(y_all)),
    batch_size=64, shuffle=True,
)

In [ ]:
EPOCHS = 20
model.train()
for epoch in range(EPOCHS):
    correct, total, loss_sum = 0, 0, 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * X_batch.size(0)
        correct += (outputs.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.size(0)
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | loss={loss_sum/total:.4f} | train_acc={correct/total:.3f}")

print("\nFinal model trained on 100% of available real data.")
print("(No held-out test set here on purpose -- Step 5 already proved generalization;")
print("this run's job is only to produce the best possible deployed model.)")

In [ ]:
# Export everything Django will need: weights + metadata, bundled into
# ONE file so there's no risk of the class-name order getting out of sync
# between training and inference later.
from google.colab import drive
drive.mount('/content/drive')

export_dir = '/content/drive/MyDrive/machine_doctor_dl'
os.makedirs(export_dir, exist_ok=True)

export_package = {
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "window_size": WINDOW_SIZE,
}
export_path = os.path.join(export_dir, "vibration_cnn_final.pt")
torch.save(export_package, export_path)

size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f"Exported to {export_path} ({size_mb:.2f} MB)")
print("\nDownload this single file from your Google Drive to your laptop --")
print("it's everything Step 7 (Django integration) needs.")